In [1]:
import torch
import torch.nn as nn
from torch import Tensor

device = "cuda"


class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size=16):
        super().__init__()
        self.conv2d = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, X:Tensor):
        X = self.conv2d(X) #shape = [batch, channel, height, width]
        X = X.flatten(start_dim=2) #shape = [batch, channel, height * width]
        return X.transpose(1,2) # shape = [batch, height * width, channel] <= nn.Transformer encoder expects like this

In [ ]:
class Vit(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, num_classes=1000, embed_dim=768,
                 depth=12, num_heads=12, ff_dim=3072, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, embed_dim, patch_size)
        cls_init = torch.randn(1,1,embed_dim) * 0.02
        self.cls_token = nn.Parameter(cls_init)
        num_patchs = (img_size // patch_size) ** 2
        pos_init = torch.randn(1, num_patchs + 1, embed_dim) * 0.02
        self.pos_embed = nn.Parameter(pos_init)
        self.dropout = nn.Dropout(p=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout,
            activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, depth)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.output = nn.Linear(embed_dim, num_classes)
    
    def forward(self, X):
        Z = self.patch_embed(X)
        cls_expd = self.cls_token.expand(Z.shape[0], -1, -1)
        Z = torch.cat((cls_expd, Z), dim=1)
        Z = Z + self.pos_embed
        Z = self.dropout(Z)
        Z = self.encoder(Z)
        Z = self.layer_norm(Z[:, 0])
        logits = self.output(Z)
        return logits
        

In [5]:
vit_model = Vit()
batch = torch.randn(4,3,224,224)
logits = vit_model(batch)
logits

tensor([[ 0.1139,  1.3270, -0.5977,  ...,  0.5041, -1.0077, -0.1764],
        [-0.5518,  1.5696, -0.7589,  ...,  0.6887, -0.9247,  0.4881],
        [-0.1853,  1.2567, -0.9403,  ...,  0.2764, -0.7416, -0.0182],
        [-0.5251,  1.3758, -0.3267,  ...,  0.2122, -0.1918,  0.1759]],
       grad_fn=<AddmmBackward0>)